# LUMA WNBA — Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lumahoops/WNBA/blob/main/notebooks/quickstart.ipynb)

Lineup-stint data for the WNBA, 2003–2026. **No download, no clone, no GitHub account.**
Press the play button on each cell in order.

Data: https://github.com/lumahoops/WNBA · DOI [10.5281/zenodo.21972004](https://doi.org/10.5281/zenodo.21972004)

## 1. Setup

Installs the package if it is not already present. Tries PyPI first, then falls back to
installing directly from GitHub, so this works even before a PyPI release.

In [ ]:
# Installs luma-wnba if needed. Tries PyPI first, then GitHub.
# Note: the PyPI attempt is expected to fail until the first PyPI release,
# so the GitHub fallback below is the normal path today. Both are fine.
import importlib, subprocess, sys

def _ensure(module="luma_wnba"):
    try:
        m = importlib.import_module(module)
        print("luma_wnba", m.__version__, "already available")
        return m
    except ImportError:
        pass
    targets = ["luma-wnba",
               "https://github.com/lumahoops/WNBA/archive/refs/heads/main.tar.gz"]
    for target in targets:
        print("installing from", target, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", target],
                       capture_output=True)
        importlib.invalidate_caches()
        try:
            m = importlib.import_module(module)
            print("OK - luma_wnba", m.__version__, "installed from", target)
            return m
        except ImportError:
            print("  not available there, trying the next source")
    raise RuntimeError(
        "Could not install luma_wnba. If you are offline or behind a proxy, "
        "download the repo from https://github.com/lumahoops/WNBA and run "
        "pip install . from inside it.")

_ensure()
print("ready")

## 2. Load a season

`SEASON` is derived from the data, not hard-coded, so this notebook keeps working as new
seasons are added. Files are fetched over the network and cached, so the second call is instant.

In [ ]:
import luma_wnba as luma

SEASON = max(luma.seasons("rs"))        # latest season in the corpus
print("available regular seasons:", luma.seasons("rs")[0], "to", SEASON)

games = luma.load_stints(SEASON)
if not games:
    raise SystemExit("No games returned for %d - the season may not have started." % SEASON)
print(len(games), "games loaded for", SEASON)

## 3. Who played the most minutes?

In [ ]:
seconds = luma.player_seconds(games)
names   = luma.load_crosswalk()

top = sorted(seconds.items(), key=lambda kv: -kv[1])[:10]
for rank, (pid, secs) in enumerate(top, 1):
    who = names.get(pid, {}).get("display_name", pid)   # fall back to the id
    print("%2d. %-24s %6.1f min" % (rank, who, secs / 60))

## 4. What a stint looks like

Each stint is `[home_five, away_five, seconds, home_points, away_points, home_tally, away_tally]`.
The two tallies count that team's offensive events during the stint.

In [ ]:
gid, date, stint = next(luma.iter_stints(games))
home, away, secs, hpts, apts, htally, atally = stint

print("game", gid, "on", date)
print("seconds", secs, "| home", hpts, "away", apts)
print()
for k, v in luma.tally_dict(htally).items():
    print("  %-12s %s" % (k, v))

## 5. Possessions and efficiency

Possessions follow Oliver's formula.

In [ ]:
def possessions(tally):
    """Oliver's possession estimate from a 16-slot tally."""
    t = tally
    return t[0] + t[2] + t[4] + 0.44 * t[6] + t[8] - t[9]

tot_poss = tot_pts = 0.0
for _gid, _date, s in luma.iter_stints(games):
    tot_poss += possessions(s[5]) + possessions(s[6])
    tot_pts  += s[3] + s[4]

print("%d league points per 100 possessions: %.1f" % (SEASON, 100 * tot_pts / tot_poss))

## 6. Player impact ratings

`ARC` combines a ridge-regularised on/off estimate with a box component, in points per 100
possessions relative to league average. `RAPM` is the pure on/off estimate with no box prior.

In [ ]:
arc = luma.load_arc(SEASON)
print("board keys:", list(arc)[:6])
print("players rated:", arc.get("n"))

## 7. Pin a version for reproducibility

Pass `ref=` a git tag so your notebook returns the same numbers forever.

```python
games = luma.load_stints(2026, ref="v1.0.0")
```

Cite the dataset with the concept DOI `10.5281/zenodo.21972004`, which always resolves to the
newest version.

## Next steps

- `SCHEMA.md` — every field
- `METRICS.md` — how ARC and RAPM are computed
- `luma.seasons("rs")` / `luma.seasons("po")` — what is available
- Terminal: `luma-wnba top 2026`